# 06.02 — Dash Integration: Profile Explorer

Orthograph's `GraphProfile` is a rich data model.  This notebook shows how
to surface one inside a **Plotly Dash** application — an interactive dashboard
that renders counts, property completeness, and cardinality distributions
directly from the profile object.

**Analogy with 06.01 (FastAPI):**  
In 06.01, `orthograph` is the *contract layer* — it validates Cypher and
defines typed query outputs that plug into FastAPI routes.  Here the analogy
is on the *observation side*: the profile is the typed data object, and Dash
is the presentation layer.  `orthograph.api.visualization.render_profile` gives
you a text table; Dash gives you an interactive browser UI.  The library does
not depend on Dash — the wiring lives entirely in application code.

**Key constraint (ADR-024):**  `src/orthograph/` must not acquire a hard or
optional dependency on Dash, Plotly, or any UI framework.  Everything shown
here lives in the consuming application layer (`notebooks/shared/dash_app.py`).

Sections:
1. Dependency check — skip gracefully if Dash is not installed
2. Profile source — choose hardcoded or live database
3. The `shared/dash_app.py` mini-package — what it does
4. Launch option A — hardcoded synthetic profile (no DB required)
5. Launch option B — live database connection
6. Exploring the UI — tabs and interactions
7. Using the profile model directly in Dash callbacks
8. Limitations and extension points

## 1. Dependency check

This notebook requires `dash`, `dash-bootstrap-components`, and `plotly`.  They
are **not** dependencies of `orthograph` itself.  The cell below attempts to
import them and halts with a clear message if any package is missing.

In [17]:
try:
    import dash
    import dash_bootstrap_components as dbc
    import plotly
except ImportError as _e:
    _msg = (
        f"Missing package: {_e.name}\n"
        "Install the UI extras with:\n"
        "    pip install dash dash-bootstrap-components plotly\n"
        "This notebook is skipped — orthograph itself does not require Dash."
    )
    print(_msg)
    raise SystemExit(_msg) from _e

print(
    f"dash {dash.__version__}  |  dbc {dbc.__version__}  |  plotly {plotly.__version__}  — OK"
)

dash 4.3.0  |  dbc 2.0.4  |  plotly 6.8.0  — OK


## 2. Profile source — choose hardcoded or live database

This demo can be driven two ways.  Set `USE_LIVE_DB = True` and fill in your
connection details to inspect a real database; leave it `False` to use the
built-in synthetic filmography profile.

The two paths produce the same `GraphProfile` type — the Dash app does not
know or care which route was used.

In [18]:
# --- Choose your profile source ---
USE_LIVE_DB = False  # set True to connect to a real database

# Live-DB settings (only read when USE_LIVE_DB = True)
DB_BACKEND = "neo4j"  # "neo4j" or "memgraph"
DB_URI = "bolt://localhost:7687"
DB_USER = "neo4j"
DB_PASSWORD = "password"

print(
    f"Profile source: {'live DB ({DB_BACKEND} @ {DB_URI})' if USE_LIVE_DB else 'hardcoded synthetic profile'}"
)

Profile source: hardcoded synthetic profile


## 3. The `shared/dash_app.py` mini-package — what it does

`notebooks/shared/dash_app.py` is a small self-contained module that accepts a
`GraphProfile` and wires it into a Dash layout.  It is intentionally minimal:
no callbacks that mutate state, no authentication, no deployment config.  It is
a demo, not a production app.

The module exposes three public names:

| Name | What it does |
|---|---|
| `create_app(profile)` | Build and return a `dash.Dash` instance from a `GraphProfile`. |
| `create_app_from_profile(profile)` | Alias for `create_app`. |
| `create_app_from_connection(uri, user, pwd, backend)` | Inspect a live DB, then call `create_app`. |

The app has three tabs:

1. **Overview** — bar chart + tables of node/relationship counts.
2. **Properties** — sortable, filterable table of completeness and type info per property.
3. **Cardinality** — degree distribution charts per relationship type.

In [19]:
# The shared/ directory is on sys.path via notebooks/conftest.py.
from shared.dash_app import (
    create_app,
    create_app_from_connection,
)


print("dash_app module imported OK")
print(
    "Available factories:",
    ["create_app", "create_app_from_profile", "create_app_from_connection"],
)

dash_app module imported OK
Available factories: ['create_app', 'create_app_from_profile', 'create_app_from_connection']


## 4. Launch option A — hardcoded synthetic profile (no DB required)

The `shared/dash_app.py` module includes a built-in synthetic filmography
profile (the same domain used across all 05.xx notebooks).  Build the profile
here inline so the notebook is transparent about what the app sees.

In [20]:
from orthograph.graph_profile.models import (
    CardinalityStats,
    ConstraintInfo,
    GraphProfile,
    NodeTypeProfile,
    PropertyProfile,
    RelationshipTypeProfile,
)


# --- Build the demo profile (filmography domain) ---
demo_profile = GraphProfile(
    source="demo — synthetic filmography profile",
    node_type_profiles={
        "Person": NodeTypeProfile(
            label="Person",
            count=120,
            property_profiles={
                "name": PropertyProfile(
                    name="name",
                    present_count=120,
                    total_count=120,
                    observed_types=["String"],
                    constraint_required=True,
                ),
                "born": PropertyProfile(
                    name="born",
                    present_count=90,
                    total_count=120,
                    observed_types=["Long"],
                    constraint_required=False,
                ),
            },
        ),
        "Movie": NodeTypeProfile(
            label="Movie",
            count=38,
            property_profiles={
                "title": PropertyProfile(
                    name="title",
                    present_count=38,
                    total_count=38,
                    observed_types=["String"],
                    constraint_required=True,
                ),
                "released": PropertyProfile(
                    name="released",
                    present_count=38,
                    total_count=38,
                    observed_types=["Long"],
                ),
                "tagline": PropertyProfile(
                    name="tagline",
                    present_count=22,
                    total_count=38,
                    observed_types=["String"],
                ),
            },
        ),
        "City": NodeTypeProfile(
            label="City",
            count=12,
            property_profiles={
                "name": PropertyProfile(
                    name="name",
                    present_count=12,
                    total_count=12,
                    observed_types=["String"],
                ),
            },
        ),
    },
    rel_type_profiles={
        "ACTED_IN": RelationshipTypeProfile(
            rel_type="ACTED_IN",
            count=253,
            source_labels={"Person"},
            target_labels={"Movie"},
            property_profiles={
                "role": PropertyProfile(
                    name="role",
                    present_count=253,
                    total_count=253,
                    observed_types=["String"],
                ),
            },
            cardinality_stats=CardinalityStats(
                count=120,
                min=1.0,
                max=8.0,
                mean=2.1,
                variance=3.5,
                histogram={
                    "1": 45,
                    "2": 30,
                    "3": 20,
                    "4": 15,
                    "5": 7,
                    "6": 2,
                    "7": 0,
                    "8": 1,
                },
            ),
        ),
        "DIRECTED": RelationshipTypeProfile(
            rel_type="DIRECTED",
            count=38,
            source_labels={"Person"},
            target_labels={"Movie"},
            cardinality_stats=CardinalityStats(
                count=30,
                min=1.0,
                max=3.0,
                mean=1.27,
                histogram={"1": 22, "2": 6, "3": 2},
            ),
        ),
        "LIVES_IN": RelationshipTypeProfile(
            rel_type="LIVES_IN",
            count=90,
            source_labels={"Person"},
            target_labels={"City"},
        ),
    },
    constraints=[
        ConstraintInfo(
            name="person_name_exists",
            constraint_type="NODE_PROPERTY_EXISTENCE",
            entity_type="NODE",
            labels=["Person"],
            properties=["name"],
        ),
        ConstraintInfo(
            name="movie_title_exists",
            constraint_type="NODE_PROPERTY_EXISTENCE",
            entity_type="NODE",
            labels=["Movie"],
            properties=["title"],
        ),
    ],
)

print("Demo profile built.")
print("  Node types      :", sorted(demo_profile.node_labels))
print("  Relationship types:", sorted(demo_profile.relationship_types))

Demo profile built.
  Node types      : ['City', 'Movie', 'Person']
  Relationship types: ['ACTED_IN', 'DIRECTED', 'LIVES_IN']


In [21]:
# --- Confirm render_profile still works on this profile (text output) ---
from orthograph.api import visualization


print(visualization.render_profile(demo_profile))

Profile: demo — synthetic filmography profile
Timestamp: 2026-06-22 15:07:11.361314

Node Types
------------------------------------------------------------
  Person (120 instances)
    name: 100% complete (120/120) [constrained] types=[String]
    born: 75% complete (90/120) [unconstrained] types=[Long]

  Movie (38 instances)
    title: 100% complete (38/38) [constrained] types=[String]
    released: 100% complete (38/38) types=[Long]
    tagline: 58% complete (22/38) types=[String]

  City (12 instances)
    name: 100% complete (12/12) types=[String]

Relationship Types
------------------------------------------------------------
  ACTED_IN (253 instances)
    sources: ['Person']
    targets: ['Movie']
    cardinality: min=1.0, max=8.0, avg=2.1, sample_size=120
    role: 100% complete (253/253) types=[String]

  DIRECTED (38 instances)
    sources: ['Person']
    targets: ['Movie']
    cardinality: min=1.0, max=3.0, avg=1.3, sample_size=30

  LIVES_IN (90 instances)
    sources: ['P

## 5. Launch option B — live database connection

When `USE_LIVE_DB = True`, the cell below inspects your database and builds the
profile.  The result is the same `GraphProfile` type, built by the same
`orthograph.api.database.inspect` call that 04.01–04.02 cover.

The Dash app factory does not know which path was taken — it receives a profile.

In [13]:
if USE_LIVE_DB:
    from shared.dash_app import create_app_from_connection

    # create_app_from_connection inspects the DB and wires the profile into Dash
    # in one call.  It returns the same app object that create_app returns.
    app = create_app_from_connection(
        uri=DB_URI,
        username=DB_USER,
        password=DB_PASSWORD,
        backend=DB_BACKEND,
        title=f"Graph Profile — {DB_URI}",
    )
    print(f"App built from live {DB_BACKEND} database at {DB_URI}")
else:
    # Use the synthetic profile built in §4
    app = create_app(
        demo_profile,
        title="Graph Profile Explorer — filmography demo",
    )
    print("App built from synthetic demo profile.")

print(f"App title: {app.title}")

App built from synthetic demo profile.
App title: Graph Profile Explorer — filmography demo


## 6. Exploring the UI — tabs and interactions

Run the cell below to start the development server.  Open
http://127.0.0.1:8050/ in your browser to see the app.

**Interrupt the kernel to stop the server** (Ctrl+C or the stop button).

The app has three tabs:

| Tab | Contents |
|---|---|
| **Overview** | Bar chart of entity counts; node-type and relationship-type tables. |
| **Properties** | Filterable, sortable table: completeness %, observed types, constraint status per property. |
| **Cardinality** | Per-relationship-type degree distribution charts (where `cardinality_stats` is available). |

> **Jupyter note:** The Dash server runs on a separate thread when called from
> a notebook.  In some environments (JupyterLab, VS Code notebooks) you may
> need `app.run(jupyter_mode='inline')` or `app.run(jupyter_mode='external')`
> to see the app inline.  Adjust `JUPYTER_MODE` below.

In [14]:
# Set JUPYTER_MODE to:
#   'external'  — open in a new browser tab (default, most compatible)
#   'inline'    — embed in the notebook output cell (JupyterLab/VS Code)
#   'tab'       — open in a new browser tab (same as external)
# Set START_SERVER = False to skip starting the server (e.g. during CI)
JUPYTER_MODE = "external"
START_SERVER = True

if START_SERVER:
    print(f"Starting Dash server at http://127.0.0.1:8050/  (mode={JUPYTER_MODE})")
    print("Interrupt the kernel to stop.")
    app.run(
        debug=True,
        use_reloader=False,  # disable reloader in notebook context
        jupyter_mode=JUPYTER_MODE,
    )
else:
    print("Server start skipped (START_SERVER=False).")
    print("Set START_SERVER=True and re-run to launch the app.")

Starting Dash server at http://127.0.0.1:8050/  (mode=external)
Interrupt the kernel to stop.
Dash app running on http://127.0.0.1:8050/


## 7. Using the profile model directly in Dash callbacks

The section above uses the pre-built layout from `shared/dash_app.py`.  This
section shows the pattern for writing your **own** callback that reads from
a `GraphProfile` directly — the model is just a Python object; there is nothing
Dash-specific about reading it.

Example: a dropdown to pick a node label, then display its property completeness
as a bar chart via a `@callback`.

In [15]:
import dash
from dash import Input, Output, dcc, html


# The profile is a plain Python object — read it however you like.
available_labels = sorted(demo_profile.node_type_profiles.keys())

callback_app = dash.Dash(
    __name__ + "_callback",
    suppress_callback_exceptions=True,
)

callback_app.layout = html.Div(
    [
        html.H2("Property completeness for a node label"),
        dcc.Dropdown(
            id="label-dropdown",
            options=[{"label": lbl, "value": lbl} for lbl in available_labels],
            value=available_labels[0] if available_labels else None,
            clearable=False,
            style={"width": "300px"},
        ),
        dcc.Graph(id="completeness-chart"),
    ]
)


@callback_app.callback(
    Output("completeness-chart", "figure"),
    Input("label-dropdown", "value"),
)
def update_completeness(label: str):
    """Read the profile model directly inside a Dash callback."""
    ntp = demo_profile.node_type_profiles.get(label)
    if ntp is None or not ntp.property_profiles:
        return {"data": [], "layout": {"title": f"No properties for {label}"}}

    props = sorted(ntp.property_profiles.keys())
    completeness = [ntp.property_profiles[p].completeness * 100 for p in props]

    return {
        "data": [
            {
                "type": "bar",
                "x": props,
                "y": completeness,
                "marker": {
                    "color": [
                        "#27ae60" if c == 100 else "#e67e22" if c >= 75 else "#c0392b"
                        for c in completeness
                    ]
                },
            }
        ],
        "layout": {
            "title": f"{label} — property completeness",
            "xaxis": {"title": "Property"},
            "yaxis": {"title": "Completeness (%)", "range": [0, 105]},
            "margin": {"t": 50, "b": 60},
        },
    }


print("Callback app built — run the next cell to start it.")

Callback app built — run the next cell to start it.


In [16]:
# Uncomment to start the callback app
callback_app.run(debug=True, port=8051, use_reloader=False, jupyter_mode=JUPYTER_MODE)

Dash app running on http://127.0.0.1:8051/


## 8. Limitations and extension points

| Area | Current state | Extension path |
|---|---|---|
| **Profile comparison** | Single profile only | Pass two profiles; diff via `compare_profiles`; colour-code changes in the table. |
| **Partitioned cardinality** | Not rendered | Add a fourth tab reading `source_partitioned_cardinality`; one row per `PartitionKey`. |
| **Live refresh** | None (static snapshot) | Add a Dash `Interval` component; call `inspect()` on each tick; store in `dcc.Store`. |
| **Comparison against definition** | Not shown | Run `compare_profile_to_definition`; colour-code violations in the property table. |
| **Export** | None | Add a `dcc.Download` button; serialize via `profile.model_dump_json()`. |
| **Authentication** | None | Use Dash Enterprise auth or a reverse proxy. |
| **Deployment** | Dev server only | Wrap `create_app` in a WSGI entry-point; deploy behind gunicorn/uvicorn. |

The `shared/dash_app.py` module is intentionally thin — it is a reference
starting point, not a complete product.  Copy it, rename it, and extend it
for your own application.